In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print("FOLDER:", dirname)
    for filename in filenames:
        print("FILE:", filename)

In [ ]:
import pandas as pd

file_path = "/kaggle/input/datasets/atharvaingle/crop-recommendation-dataset/Crop_recommendation.csv"

df = pd.read_csv(file_path)

df.head()

In [ ]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Number of crop classes:", df["label"].nunique())

print("\nCrop classes:")
print(df["label"].unique())

print("\nSamples per crop:")
print(df["label"].value_counts())

In [ ]:
#visualize

plt.figure(figsize=(12, 6))

sns.countplot(data=df, x="label", order=df["label"].value_counts().index)

plt.xticks(rotation=45)
plt.xlabel("Crop")
plt.ylabel("Number of Samples")
plt.title("Crop Class Distribution")
plt.tight_layout()
plt.show()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
crop_means = df.groupby("label")[
    ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
].mean()

crop_means

In [ ]:
import matplotlib.pyplot as plt

features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]

df[features].hist(figsize=(14, 9), bins=20)

plt.suptitle("Distribution of Crop Recommendation Features", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[features].corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap="coolwarm", aspect="auto")
plt.colorbar()

plt.xticks(range(len(features)), features, rotation=45)
plt.yticks(range(len(features)), features)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
#eda done

In [ ]:
X = df.drop("label", axis=1)
y = df["label"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

In [ ]:
#train test splittt
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y #crop classes ka proportion maintain rahega
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
#random forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

In [ ]:
# SHAP explainability for Crop Recommendation

import shap
import pandas as pd

# Load crop dataset separately
crop_df = pd.read_csv(
    "/kaggle/input/datasets/atharvaingle/crop-recommendation-dataset/Crop_recommendation.csv"
)

# Use exactly the features used by the trained Random Forest
crop_features = rf_model.feature_names_in_

X_crop_shap = crop_df[crop_features].sample(200, random_state=42)

# Create SHAP explainer
explainer = shap.TreeExplainer(rf_model)

# Calculate SHAP values
shap_values = explainer.shap_values(X_crop_shap)

print("SHAP values calculated successfully!")
print("Features:", list(crop_features))

In [ ]:
# Overall SHAP Feature Importance for Crop Recommendation

import numpy as np
import matplotlib.pyplot as plt

# Combine SHAP values across all crop classes
if isinstance(shap_values, list):
    shap_array = np.stack(shap_values, axis=0)
    mean_shap = np.mean(np.abs(shap_array), axis=0)
else:
    # Newer SHAP versions may return a 3D array
    mean_shap = np.mean(np.abs(shap_values), axis=-1)

# Calculate average importance of each feature
feature_importance = np.mean(mean_shap, axis=0)

# Create feature importance table
importance_df = pd.DataFrame({
    "Feature": X_crop_shap.columns,
    "Mean |SHAP Value|": feature_importance
}).sort_values("Mean |SHAP Value|", ascending=True)

# Plot
plt.figure(figsize=(8, 5))
plt.barh(
    importance_df["Feature"],
    importance_df["Mean |SHAP Value|"]
)
plt.xlabel("Mean |SHAP Value|")
plt.ylabel("Feature")
plt.title("SHAP Feature Importance - Crop Recommendation")
plt.tight_layout()
plt.show()

In [ ]:
# Local SHAP Explanation for a Specific Farmer Input

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Example farmer input
farmer_input = pd.DataFrame([{
    "N": 90,
    "P": 42,
    "K": 43,
    "temperature": 21.0,
    "humidity": 82.0,
    "ph": 6.5,
    "rainfall": 203.0
}])

# Predict crop
predicted_crop = rf_model.predict(farmer_input)[0]

print("Recommended Crop:", predicted_crop)

# Calculate SHAP values for this input
local_shap = explainer.shap_values(farmer_input)

# Find the predicted crop class
crop_index = list(rf_model.classes_).index(predicted_crop)

# Handle SHAP output format
if isinstance(local_shap, list):
    crop_shap_values = local_shap[crop_index][0]
else:
    if local_shap.ndim == 3:
        crop_shap_values = local_shap[0, :, crop_index]
    else:
        crop_shap_values = local_shap[0]

# Create explanation table
local_explanation = pd.DataFrame({
    "Feature": farmer_input.columns,
    "Input Value": farmer_input.iloc[0].values,
    "SHAP Contribution": crop_shap_values
})

local_explanation["Absolute Contribution"] = (
    local_explanation["SHAP Contribution"].abs()
)

local_explanation = local_explanation.sort_values(
    "Absolute Contribution",
    ascending=False
)

print("\nTop Factors Influencing the Prediction:")
display(local_explanation)

In [ ]:
# Local SHAP Explanation Plot

plot_df = local_explanation.sort_values("SHAP Contribution")

plt.figure(figsize=(8, 5))

plt.barh(
    plot_df["Feature"],
    plot_df["SHAP Contribution"]
)

plt.axvline(0, linewidth=1)
plt.xlabel("SHAP Contribution")
plt.ylabel("Feature")
plt.title(f"Why was {predicted_crop.title()} recommended?")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
#conf-matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)

plt.figure(figsize=(12, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=rf_model.classes_
)

disp.plot(
    xticks_rotation=90,
    cmap="Blues",
    values_format="d"
)

plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
#feature importance
import pandas as pd
import matplotlib.pyplot as plt

importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

plt.figure(figsize=(9, 5))
importance.sort_values().plot(kind="barh")

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
#top3 from existing
import numpy as np

# Get probability of every crop
probabilities = rf_model.predict_proba(X_test)

# Take the first test sample
sample_index = 0

crop_probabilities = pd.Series(
    probabilities[sample_index],
    index=rf_model.classes_
).sort_values(ascending=False)

print("Top-3 Recommended Crops:")
print(crop_probabilities.head(3))

In [ ]:
import pandas as pd

farmer_input = pd.DataFrame({
    "N": [90],
    "P": [42],
    "K": [43],
    "temperature": [20.8],
    "humidity": [82],
    "ph": [6.5],
    "rainfall": [202]
})

probabilities = rf_model.predict_proba(farmer_input)[0]

results = pd.DataFrame({
    "Crop": rf_model.classes_,
    "Probability": probabilities
})
results = results[results["Probability"] > 0]

top3 = results.sort_values(
    "Probability",
    ascending=False
).head(3)

top3["Probability"] = (top3["Probability"] * 100).round(2)

top3

In [ ]:
#irrigTIONNNNN

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv")

df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
import pandas as pd

# Load irrigation dataset
df = pd.read_csv(
    "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"
)

# Basic information
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

print("\nMissing Values:")
display(df.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of irrigation requirement
print("Irrigation Need Distribution:")
print(df["Irrigation_Need"].value_counts())

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Irrigation_Need")
plt.title("Distribution of Irrigation Requirement")
plt.xlabel("Irrigation Need")
plt.ylabel("Number of Samples")
plt.show()

In [ ]:
# Statistical summary of numerical features
df.describe().T

In [ ]:
# Correlation between numerical features
numeric_df = df.select_dtypes(include="number")

plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Numerical Features")
plt.show()

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include="object").columns

print("Categorical Columns:")
print(categorical_cols.tolist())

In [ ]:
#Soil Type vs Irrigation Need
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="Soil_Type",
    hue="Irrigation_Need"
)

plt.title("Irrigation Requirement by Soil Type")
plt.xlabel("Soil Type")
plt.ylabel("Number of Samples")
plt.legend(title="Irrigation Need")
plt.tight_layout()
plt.show()

In [ ]:
#Season vs Irrigation Need
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="Season",
    hue="Irrigation_Need"
)

plt.title("Irrigation Requirement by Season")
plt.xlabel("Season")
plt.ylabel("Number of Samples")
plt.legend(title="Irrigation Need")
plt.tight_layout()
plt.show()

In [ ]:
#Crop Type vs Irrigation Need
plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="Crop_Type",
    hue="Irrigation_Need"
)

plt.title("Irrigation Requirement by Crop Type")
plt.xlabel("Crop Type")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Soil Moisture vs Irrigation Need
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df,
    x="Irrigation_Need",
    y="Soil_Moisture"
)

plt.title("Soil Moisture Distribution by Irrigation Requirement")
plt.xlabel("Irrigation Need")
plt.ylabel("Soil Moisture")
plt.tight_layout()
plt.show()

In [ ]:
#Crop Growth Stage vs Irrigation Need
plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="Crop_Growth_Stage",
    hue="Irrigation_Need"
)

plt.title("Irrigation Requirement by Crop Growth Stage")
plt.xlabel("Crop Growth Stage")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Rainfall vs Irrigation Need
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df,
    x="Irrigation_Need",
    y="Rainfall_mm"
)

plt.title("Rainfall Distribution by Irrigation Requirement")
plt.xlabel("Irrigation Need")
plt.ylabel("Rainfall (mm)")
plt.tight_layout()
plt.show()

In [ ]:
#set target n features
X = df.drop("Irrigation_Need", axis=1)
y = df["Irrigation_Need"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

print("\nTarget Classes:")
print(y.unique())

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numeric_cols)

In [ ]:
#Encoding + Model Pipeline..............OneHotEncoder + ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numerical", "passthrough", numeric_cols)
    ]
)

# Random Forest model
irrigation_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

# Train
irrigation_model.fit(X_train, y_train)

print("Irrigation model trained successfully!")

In [ ]:
#model eval
from sklearn.metrics import accuracy_score, classification_report

# Predictions
y_pred_irr = irrigation_model.predict(X_test)

# Accuracy
accuracy_irr = accuracy_score(y_test, y_pred_irr)

print("Irrigation Model Accuracy:", accuracy_irr)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_irr))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(
    y_test,
    y_pred_irr,
    labels=["Low", "Medium", "High"]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Low", "Medium", "High"]
)

disp.plot(values_format="d")

plt.title("Irrigation Requirement - Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

irrigation_model_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            class_weight="balanced_subsample",
            min_samples_leaf=2
        ))
    ]
)

irrigation_model_v2.fit(X_train, y_train)

y_pred_irr_v2 = irrigation_model_v2.predict(X_test)

print("New Irrigation Model Accuracy:",
      accuracy_score(y_test, y_pred_irr_v2))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_irr_v2))

In [ ]:
# Fast SHAP Explainability for Irrigation Model

import shap
import pandas as pd
import numpy as np

# Get trained preprocessing and Random Forest
preprocessor = irrigation_model_v2.named_steps["preprocessor"]
rf_irrigation = irrigation_model_v2.named_steps["classifier"]

# Use only 10 samples
X_test_small = X_test.iloc[:10].copy()

# Transform using the same preprocessing
X_small_transformed = preprocessor.transform(X_test_small)

if hasattr(X_small_transformed, "toarray"):
    X_small_transformed = X_small_transformed.toarray()

# Feature names after encoding
feature_names = preprocessor.get_feature_names_out()

X_small_shap = pd.DataFrame(
    X_small_transformed,
    columns=feature_names
)

# SHAP explainer
irrigation_explainer = shap.TreeExplainer(
    rf_irrigation
)

# Approximate SHAP calculation
irrigation_shap_values = irrigation_explainer.shap_values(
    X_small_shap,
    approximate=True
)

print("Irrigation SHAP calculated successfully!")
print("Samples:", len(X_small_shap))
print("Encoded features:", len(feature_names))

In [ ]:
# Overall SHAP Feature Importance - Irrigation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Handle SHAP output for the 3 irrigation classes
if isinstance(irrigation_shap_values, list):
    shap_array = np.stack(irrigation_shap_values, axis=0)
    mean_shap = np.mean(np.abs(shap_array), axis=(0, 1))
else:
    mean_shap = np.mean(np.abs(irrigation_shap_values), axis=(0, 2))

# Create feature importance table
irrigation_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean |SHAP Value|": mean_shap
}).sort_values(
    "Mean |SHAP Value|",
    ascending=True
)

# Plot top 15 features
top_features = irrigation_importance.tail(15)

plt.figure(figsize=(9, 6))
plt.barh(
    top_features["Feature"],
    top_features["Mean |SHAP Value|"]
)

plt.xlabel("Mean |SHAP Value|")
plt.ylabel("Feature")
plt.title("SHAP Feature Importance - Irrigation Recommendation")
plt.tight_layout()
plt.show()

In [ ]:
# Local SHAP Explanation - Irrigation

# Take one test sample
sample = X_test.iloc[[0]].copy()

# Actual irrigation prediction
predicted_irrigation = irrigation_model_v2.predict(sample)[0]

print("Predicted Irrigation Need:", predicted_irrigation)

# Transform sample using the trained preprocessor
sample_transformed = preprocessor.transform(sample)

if hasattr(sample_transformed, "toarray"):
    sample_transformed = sample_transformed.toarray()

sample_shap = pd.DataFrame(
    sample_transformed,
    columns=feature_names
)

# Calculate SHAP for this one sample
local_irrigation_shap = irrigation_explainer.shap_values(
    sample_shap,
    approximate=True
)

# Find predicted class
class_index = list(rf_irrigation.classes_).index(predicted_irrigation)

# Extract SHAP values for predicted class
if isinstance(local_irrigation_shap, list):
    local_values = local_irrigation_shap[class_index][0]
else:
    if local_irrigation_shap.ndim == 3:
        local_values = local_irrigation_shap[0, :, class_index]
    else:
        local_values = local_irrigation_shap[0]

# Create explanation table
local_irrigation_explanation = pd.DataFrame({
    "Feature": feature_names,
    "SHAP Contribution": local_values
})

local_irrigation_explanation["Absolute Contribution"] = (
    local_irrigation_explanation["SHAP Contribution"].abs()
)

local_irrigation_explanation = (
    local_irrigation_explanation
    .sort_values("Absolute Contribution", ascending=False)
    .head(10)
)

print("\nTop Factors Influencing Irrigation Prediction:")
display(local_irrigation_explanation)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_v2 = confusion_matrix(
    y_test,
    y_pred_irr_v2,
    labels=["Low", "Medium", "High"]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_v2,
    display_labels=["Low", "Medium", "High"]
)

disp.plot(values_format="d")

plt.title("Improved Irrigation Model - Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
#Irrigation recommendation on wt  basis ?which feature among soil moisture, rainfall, previous irrigation, temperature etc. are influential 
# Feature importance of the improved irrigation model

rf = irrigation_model_v2.named_steps["classifier"]
preprocessor_fitted = irrigation_model_v2.named_steps["preprocessor"]

feature_names = preprocessor_fitted.get_feature_names_out()

importance = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print("Top 15 Important Features:")
display(importance.head(15))

In [ ]:
plt.figure(figsize=(10, 6))

importance.head(15).sort_values().plot(kind="barh")

plt.title("Top 15 Feature Importances - Irrigation Model")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
#The irrigation model primarily considers soil moisture, rainfall, temperature and wind speed, along with crop growth stage and other environmental and field-related factors.
#Among these, soil moisture was the most influential feature in our trained model.

In [ ]:
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].unique())

In [ ]:
farmer_irrigation_input = pd.DataFrame({
    "Soil_Type": ["Loamy"],
    "Soil_pH": [6.5],
    "Soil_Moisture": [25],
    "Organic_Carbon": [1.5],
    "Electrical_Conductivity": [0.5],
    "Temperature_C": [28],
    "Humidity": [65],
    "Rainfall_mm": [50],
    "Sunlight_Hours": [8],
    "Wind_Speed_kmh": [10],
    "Crop_Type": ["Rice"],
    "Crop_Growth_Stage": ["Vegetative"],
    "Season": ["Kharif"],
    "Irrigation_Type": ["Drip"],
    "Water_Source": ["Groundwater"],
    "Field_Area_hectare": [1],
    "Mulching_Used": ["Yes"],
    "Previous_Irrigation_mm": [20],
    "Region": ["North"]
})

prediction = irrigation_model_v2.predict(farmer_irrigation_input)

print("Predicted Irrigation Requirement:", prediction[0])

In [ ]:
irrigation_prob = irrigation_model_v2.predict_proba(
    farmer_irrigation_input
)[0]

irrigation_classes = irrigation_model_v2.classes_

irrigation_result = pd.DataFrame({
    "Irrigation Level": irrigation_classes,
    "Probability (%)": (irrigation_prob * 100).round(2)
}).sort_values(
    "Probability (%)",
    ascending=False
).reset_index(drop=True)

print("Irrigation Recommendation:")
display(irrigation_result)

In [ ]:
#plant village disease model

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print("FOLDER:", dirname)
    for filename in filenames[:10]:
        print("FILE:", filename)

In [ ]:
#count total images and classes
import os

train_path = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train"
val_path = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/val"

train_classes = sorted(os.listdir(train_path))
val_classes = sorted(os.listdir(val_path))

print("Number of Classes:", len(train_classes))
print("\nClasses:")
for i, cls in enumerate(train_classes):
    print(i, cls)

print("\nTotal Training Images:",
      sum(len(files) for _, _, files in os.walk(train_path)))

print("Total Validation Images:",
      sum(len(files) for _, _, files in os.walk(val_path)))

In [ ]:
#image preprocessing + model
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    val_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Training samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)
print("Number of classes:", train_generator.num_classes)

In [ ]:
import tensorflow as tf

print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
#lightweight CNN model
import tensorflow as tf
from tensorflow.keras import layers, models

cnn_model = models.Sequential([
    
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(38, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
#Train the model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

history = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=3,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
#evaluate the trained model properly and then decide whether another few epochs are worthwhile.
test_loss, test_accuracy = cnn_model.evaluate(
    val_generator,
    verbose=1
)

print("Validation Loss:", round(test_loss, 4))
print("Validation Accuracy:", round(test_accuracy * 100, 2), "%")

In [ ]:
#classification report
from sklearn.metrics import classification_report
import numpy as np

# Reset generator
val_generator.reset()

# Predict all validation images
predictions = cnn_model.predict(val_generator, verbose=1)

# Convert probabilities to class numbers
y_pred = np.argmax(predictions, axis=1)
y_true = val_generator.classes

# Get class names
class_names = list(val_generator.class_indices.keys())

# Classification report
report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=3
)

print(report)

In [ ]:
#train 2 more epochs,
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

history_2 = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    initial_epoch=3,
    epochs=5,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
# Final evaluation of the trained Disease Detection CNN

val_generator.reset()

final_loss, final_accuracy = cnn_model.evaluate(
    val_generator,
    verbose=1
)

print("Final Validation Loss:", round(final_loss, 4))
print("Final Validation Accuracy:", round(final_accuracy * 100, 2), "%")

In [ ]:
# Disease Detection CNN – Confusion Matrix

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

val_generator.reset()

predictions = cnn_model.predict(val_generator, verbose=1)

y_pred = np.argmax(predictions, axis=1)
y_true = val_generator.classes

class_names = list(val_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(18, 18))
plt.imshow(cm)
plt.title("Confusion Matrix - Plant Disease Detection")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks(
    ticks=np.arange(len(class_names)),
    labels=class_names,
    rotation=90,
    fontsize=7
)
plt.yticks(
    ticks=np.arange(len(class_names)),
    labels=class_names,
    fontsize=7
)

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
# Disease Detection CNN – Final Classification Report

from sklearn.metrics import classification_report

val_generator.reset()

predictions = cnn_model.predict(val_generator, verbose=1)

y_pred = np.argmax(predictions, axis=1)
y_true = val_generator.classes

class_names = list(val_generator.class_indices.keys())

print("Classification Report:\n")

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=3,
    zero_division=0
))

In [ ]:
# Save the trained Disease Detection CNN

model_path = "/kaggle/working/plant_disease_cnn_128.keras"

cnn_model.save(model_path)

print("Model saved successfully!")
print("Path:", model_path)

In [ ]:
# Test Disease Detection CNN on a single PlantVillage leaf image

import os
import random
import numpy as np
from tensorflow.keras.preprocessing import image

# Select a random class
random_class = random.choice(list(val_generator.class_indices.keys()))

# Select an image from that class
class_folder = os.path.join(val_path, random_class)
image_file = random.choice([
    f for f in os.listdir(class_folder)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

image_path = os.path.join(class_folder, image_file)

# Load and preprocess image
img = image.load_img(image_path, target_size=(128, 128))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Prediction
prediction = cnn_model.predict(img_array, verbose=0)

predicted_index = np.argmax(prediction[0])
predicted_class = class_names[predicted_index]
confidence = prediction[0][predicted_index] * 100

print("Actual Class     :", random_class)
print("Predicted Class  :", predicted_class)
print("Confidence       :", round(confidence, 2), "%")

In [ ]:
# Test the CNN on 5 random validation images

import os
import random
import numpy as np
from tensorflow.keras.preprocessing import image

for i in range(5):

    random_class = random.choice(list(val_generator.class_indices.keys()))

    class_folder = os.path.join(val_path, random_class)

    image_file = random.choice([
        f for f in os.listdir(class_folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    image_path = os.path.join(class_folder, image_file)

    img = image.load_img(image_path, target_size=(128, 128))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = cnn_model.predict(img_array, verbose=0)

    predicted_index = np.argmax(prediction[0])
    predicted_class = class_names[predicted_index]
    confidence = prediction[0][predicted_index] * 100

    print(f"\nTest {i+1}")
    print("Actual     :", random_class)
    print("Predicted  :", predicted_class)
    print("Confidence :", round(confidence, 2), "%")

In [ ]:
# Disease Detection – Top-3 Predictions

def predict_disease_top3(image_path):

    img = image.load_img(
        image_path,
        target_size=(128, 128)
    )

    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = cnn_model.predict(img_array, verbose=0)[0]

    top_indices = np.argsort(prediction)[::-1][:3]

    print("Top-3 Disease Predictions:\n")

    for rank, index in enumerate(top_indices, start=1):
        disease = class_names[index]
        confidence = prediction[index] * 100

        print(
            f"{rank}. {disease} - {confidence:.2f}%"
        )

In [ ]:
# Test Top-3 Disease Prediction

import os
import random

# Select Peach bacterial spot class
test_class = "Peach___Bacterial_spot"

class_folder = os.path.join(val_path, test_class)

image_file = random.choice([
    f for f in os.listdir(class_folder)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

test_image_path = os.path.join(class_folder, image_file)

print("Actual Class:", test_class)
print("Image:", image_file)

predict_disease_top3(test_image_path)

In [ ]:
# Continue Disease CNN Training
# Previous training completed: 5 epochs
# Best validation accuracy so far: 67.48%
# Continue training to improve disease classification performance

history_3 = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    initial_epoch=5,
    epochs=8,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
# Final Disease CNN Evaluation
# Total training completed: 8 epochs
# Best validation accuracy: 78.40%

val_generator.reset()

final_loss, final_accuracy = cnn_model.evaluate(
    val_generator,
    verbose=1
)

print("Final Validation Loss:", round(final_loss, 4))
print("Final Validation Accuracy:", round(final_accuracy * 100, 2), "%")

In [125]:
# Save final trained Disease Detection CNN

final_model_path = "/kaggle/working/plant_disease_cnn_final.keras"

cnn_model.save(final_model_path)

print("Final model saved successfully!")
print("Path:", final_model_path)

Final model saved successfully!
Path: /kaggle/working/plant_disease_cnn_final.keras


In [126]:
# Test final trained CNN on Peach Bacterial Spot

test_class = "Peach___Bacterial_spot"

class_folder = os.path.join(val_path, test_class)

image_file = random.choice([
    f for f in os.listdir(class_folder)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

test_image_path = os.path.join(class_folder, image_file)

print("Actual Class:", test_class)
print("Image:", image_file)

predict_disease_top3(test_image_path)

Actual Class: Peach___Bacterial_spot
Image: efc7e19f-df8a-41d8-8f58-08c90017418a___Rut._Bact.S 1398.JPG
Top-3 Disease Predictions:

1. Peach___Bacterial_spot - 23.99%
2. Grape___Black_rot - 23.94%
3. Tomato___Tomato_Yellow_Leaf_Curl_Virus - 18.44%


In [ ]:
# Continue Disease CNN Training
# Previous training: 8 epochs
# Best validation accuracy so far: 78.40%
# Continue up to 11 epochs to improve classification

history_4 = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    initial_epoch=8,
    epochs=11,
    callbacks=[early_stop, reduce_lr]
)

Epoch 9/11
 192/1358 ━━━━━━━━━━━━━━━━━━━━ 4:38 239ms/step - accuracy: 0.7627 - loss: 0.7554

In [ ]:
# Save the best Disease Detection CNN
# Best validation accuracy achieved: 78.40%

final_model_path = "/kaggle/working/plant_disease_cnn_final.keras"

cnn_model.save(final_model_path)

print("Best Disease CNN saved successfully!")
print("Validation Accuracy: 78.40%")
print("Path:", final_model_path)

In [ ]:
# Final Disease Detection – Classification Report
# Best model weights restored by EarlyStopping

val_generator.reset()

predictions = cnn_model.predict(val_generator, verbose=1)

y_pred = np.argmax(predictions, axis=1)
y_true = val_generator.classes

class_names = list(val_generator.class_indices.keys())

print("Final Classification Report:\n")

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=3,
    zero_division=0
))

In [ ]:
# Confirm final CNN performance after continued training

val_generator.reset()

loss, accuracy = cnn_model.evaluate(
    val_generator,
    verbose=1
)

print("Confirmed Validation Loss:", round(loss, 4))
print("Confirmed Validation Accuracy:", round(accuracy * 100, 2), "%")

In [ ]:
# Save the final Disease Detection CNN
# Final confirmed validation accuracy: 75.33%

final_model_path = "/kaggle/working/plant_disease_cnn_final.keras"

cnn_model.save(final_model_path)

print("Final Disease CNN saved successfully!")
print("Final Validation Accuracy: 75.33%")
print("Path:", final_model_path)

In [ ]:
# Final Disease Prediction Test
# Input: Leaf Image
# Output: Top-3 Disease Predictions with Confidence

import os
import random
import numpy as np
from tensorflow.keras.preprocessing import image

# Select a random validation class
test_class = random.choice(class_names)

class_folder = os.path.join(val_path, test_class)

image_file = random.choice([
    f for f in os.listdir(class_folder)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

test_image_path = os.path.join(class_folder, image_file)

print("Actual Class:", test_class)
print("Image:", image_file)
print()

predict_disease_top3(test_image_path)

In [ ]:
# Disease Health Guidance Layer
# Maps predicted disease to simple agricultural guidance

health_guidance = {
    "Corn_(maize)___Common_rust_": 
        "Monitor the crop regularly and remove severely affected leaves. Maintain proper field ventilation and avoid excessive moisture on foliage.",

    "Apple___Apple_scab": 
        "Remove infected leaves and fruits and maintain good orchard sanitation. Avoid prolonged leaf wetness where possible.",

    "Potato___Late_blight": 
        "Remove severely infected plant material and monitor nearby plants. Avoid prolonged leaf wetness and excessive irrigation.",

    "Tomato___Early_blight": 
        "Remove affected leaves, maintain proper spacing and avoid overhead watering. Monitor the crop regularly.",

    "Tomato___Late_blight": 
        "Remove severely infected plant material and improve air circulation. Avoid overhead irrigation and monitor disease spread."
}

def get_health_guidance(disease):

    if disease in health_guidance:
        return health_guidance[disease]

    return "Monitor the crop regularly, remove severely affected plant material, maintain proper field sanitation, and consult an agricultural expert for treatment-specific advice."

In [ ]:
# Final Post-Sowing Disease Detection
# Leaf Image → CNN → Disease Prediction → Confidence → Guidance

def final_disease_detection(image_path):

    # Load and preprocess image
    img = image.load_img(
        image_path,
        target_size=(128, 128)
    )

    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # CNN prediction
    prediction = cnn_model.predict(img_array, verbose=0)[0]

    # Top-3 predictions
    top_indices = np.argsort(prediction)[::-1][:3]

    top_index = top_indices[0]
    predicted_disease = class_names[top_index]
    confidence = prediction[top_index] * 100

    print("========================================")
    print("      PLANT DISEASE DETECTION")
    print("========================================")

    print("\nPredicted Disease:")
    print(predicted_disease)

    print("Confidence:", round(confidence, 2), "%")

    print("\nTop-3 Predictions:")
    
    for rank, index in enumerate(top_indices, start=1):
        print(
            f"{rank}. {class_names[index]} - "
            f"{prediction[index] * 100:.2f}%"
        )

    print("\nHealth Guidance:")
    print(get_health_guidance(predicted_disease))

    print("========================================")

In [ ]:
final_disease_detection(test_image_path)

In [ ]:
import os

model_path = "/kaggle/working/plant_disease_cnn_final.keras"

print("Model file exists:", os.path.exists(model_path))

In [ ]:
# Save the trained CNN model
model_path = "/kaggle/working/plant_disease_cnn_final.keras"

cnn_model.save(model_path)

print("Model saved successfully!")
print(model_path)

In [ ]:
# Check CNN layers for Grad-CAM

for layer in cnn_model.layers:
    print(layer.name, layer.__class__.__name__)

In [ ]:
# Grad-CAM for Plant Disease Detection

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Get one validation image
img_array, true_label = next(val_generator)

sample_image = img_array[0]
input_image = np.expand_dims(sample_image, axis=0)

# Make sure the trained model is called
predictions = cnn_model(input_image, training=False)

# Use model.inputs instead of model.input
grad_model = tf.keras.models.Model(
    inputs=cnn_model.inputs,
    outputs=[
        cnn_model.get_layer("conv2d_2").output,
        cnn_model.outputs[0]
    ]
)

# Calculate gradients
with tf.GradientTape() as tape:
    conv_outputs, predictions = grad_model(input_image, training=False)

    predicted_class = tf.argmax(predictions[0])
    class_output = predictions[:, predicted_class]

gradients = tape.gradient(class_output, conv_outputs)

# Average gradients
pooled_gradients = tf.reduce_mean(
    gradients,
    axis=(0, 1, 2)
)

conv_outputs = conv_outputs[0]

# Generate heatmap
heatmap = conv_outputs @ pooled_gradients[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

heatmap = tf.maximum(heatmap, 0)
heatmap /= tf.maximum(
    tf.reduce_max(heatmap),
    1e-8
)

heatmap = heatmap.numpy()

# Class names
class_names = list(val_generator.class_indices.keys())

predicted_disease = class_names[int(predicted_class)]
true_disease = class_names[np.argmax(true_label[0])]

print("True Disease:", true_disease)
print("Predicted Disease:", predicted_disease)

# Display Grad-CAM
plt.figure(figsize=(7, 6))

plt.imshow(sample_image)
plt.imshow(
    heatmap,
    alpha=0.5,
    cmap="jet"
)

plt.axis("off")
plt.title(f"Grad-CAM: {predicted_disease}")
plt.show()

In [ ]:
# Find a Correctly Classified Image for Grad-CAM

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Reset validation generator
val_generator.reset()

found = False

for images, labels in val_generator:
    
    predictions = cnn_model.predict(images, verbose=0)
    
    predicted_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(labels, axis=1)
    
    # Find correctly predicted image
    correct_indices = np.where(predicted_classes == true_classes)[0]
    
    if len(correct_indices) > 0:
        idx = correct_indices[0]
        
        sample_image = images[idx]
        true_class = true_classes[idx]
        predicted_class = predicted_classes[idx]
        confidence = predictions[idx][predicted_class] * 100
        
        found = True
        break

if found:
    class_names = list(val_generator.class_indices.keys())
    
    print("True Disease:", class_names[true_class])
    print("Predicted Disease:", class_names[predicted_class])
    print(f"Confidence: {confidence:.2f}%")
else:
    print("No correctly classified image found.")

In [ ]:
# Clean Grad-CAM for the correctly predicted image

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Prepare the selected image
input_image = np.expand_dims(sample_image, axis=0)

# Build Grad-CAM model using model.inputs
grad_model = tf.keras.models.Model(
    inputs=cnn_model.inputs,
    outputs=[
        cnn_model.get_layer("conv2d_2").output,
        cnn_model.outputs[0]
    ]
)

# Calculate gradients
with tf.GradientTape() as tape:
    conv_outputs, predictions = grad_model(input_image, training=False)

    predicted_class = tf.argmax(predictions[0])
    class_output = predictions[:, predicted_class]

gradients = tape.gradient(class_output, conv_outputs)

# Average gradients
pooled_gradients = tf.reduce_mean(
    gradients,
    axis=(0, 1, 2)
)

conv_outputs = conv_outputs[0]

# Create heatmap
heatmap = conv_outputs @ pooled_gradients[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

heatmap = tf.maximum(heatmap, 0)
heatmap /= tf.maximum(tf.reduce_max(heatmap), 1e-8)

# Display
class_names = list(val_generator.class_indices.keys())
predicted_disease = class_names[int(predicted_class)]

plt.figure(figsize=(7, 6))
plt.imshow(sample_image)
plt.imshow(heatmap.numpy(), alpha=0.5, cmap="jet")
plt.axis("off")
plt.title(
    f"Grad-CAM: {predicted_disease}\n"
    f"Confidence: {confidence:.2f}%"
)
plt.show()

In [ ]:
# OpenWeather API - Connection Test

from kaggle_secrets import UserSecretsClient
import requests

# Get API key securely from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("OPENWEATHER_API_KEY")

# Test location: Chennai
latitude = 13.0827
longitude = 80.2707

url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "lat": latitude,
    "lon": longitude,
    "appid": api_key,
    "units": "metric"
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)

if response.status_code == 200:
    weather_data = response.json()
    print("Weather API connected successfully!")
    print("Location:", weather_data["name"])
    print("Temperature:", weather_data["main"]["temp"], "°C")
    print("Humidity:", weather_data["main"]["humidity"], "%")
    print("Weather:", weather_data["weather"][0]["description"])
    print("Wind Speed:", weather_data["wind"]["speed"], "m/s")
else:
    print("API Error:", response.text)

In [ ]:
def get_weather(latitude, longitude):
    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "lat": latitude,
        "lon": longitude,
        "appid": api_key,
        "units": "metric"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        raise Exception(f"Weather API Error: {response.text}")

    data = response.json()

    return {
        "location": data["name"],
        "temperature": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "rainfall": data.get("rain", {}).get("1h", 0),
        "weather": data["weather"][0]["description"],
        "wind_speed": data["wind"]["speed"]
    }

In [ ]:
weather = get_weather(13.0827, 80.2707)

print("Weather Data:")
print(weather)

In [ ]:
temperature = weather["temperature"]
humidity = weather["humidity"]
rainfall = weather["rainfall"]
wind_speed = weather["wind_speed"]

print("Temperature:", temperature, "°C")
print("Humidity:", humidity, "%")
print("Rainfall:", rainfall, "mm")
print("Wind Speed:", wind_speed, "m/s")

In [ ]:
# Calculate weather suitability score based on current weather conditions
def weather_suitability(temperature, humidity, rainfall):
    
    # Start with maximum suitability score
    score = 100

    # Reduce score if temperature is outside the suitable range
    if temperature < 15 or temperature > 35:
        score -= 30
    elif temperature < 20 or temperature > 30:
        score -= 15

    # Reduce score if humidity is too low or too high
    if humidity < 30 or humidity > 90:
        score -= 20
    elif humidity < 40 or humidity > 80:
        score -= 10

    # Reduce score when there is no rainfall
    if rainfall == 0:
        score -= 10

    # Ensure the final score remains between 0 and 100
    return max(0, score)


# Calculate suitability using the live weather data
score = weather_suitability(
    temperature,
    humidity,
    rainfall
)

# Display the weather suitability score
print("Weather Suitability Score:", score, "/ 100")

In [ ]:
# Define approximate weather requirements for the crops
# These ranges are used to calculate crop-specific weather suitability

crop_weather_requirements = {
    
    # Weather requirements for Rice
    "rice": {
        "temperature": (20, 35),
        "humidity": (60, 90),
        "rainfall": (150, 300)
    },

    # Weather requirements for Jute
    "jute": {
        "temperature": (24, 35),
        "humidity": (60, 90),
        "rainfall": (150, 300)
    },

    # Weather requirements for Pomegranate
    "pomegranate": {
        "temperature": (20, 35),
        "humidity": (40, 70),
        "rainfall": (50, 100)
    },

    # Weather requirements for Maize
    "maize": {
        "temperature": (18, 32),
        "humidity": (50, 80),
        "rainfall": (50, 150)
    },

    # Weather requirements for Wheat
    "wheat": {
        "temperature": (10, 25),
        "humidity": (40, 70),
        "rainfall": (30, 100)
    }
}

# Confirm that the crop weather requirements were created
print("Crop weather requirements updated successfully!")
print("Number of crops:", len(crop_weather_requirements))

In [ ]:
# Calculate a crop-specific weather suitability score
# based on how close the current conditions are to the crop's requirements

def crop_weather_score(crop, temperature, humidity, rainfall):

    # Convert crop name to lowercase for matching
    crop = crop.lower()

    # Check whether weather requirements are available
    if crop not in crop_weather_requirements:
        return None

    # Get the weather requirements for the selected crop
    requirements = crop_weather_requirements[crop]

    # Extract minimum and maximum values
    temp_min, temp_max = requirements["temperature"]
    humidity_min, humidity_max = requirements["humidity"]
    rainfall_min, rainfall_max = requirements["rainfall"]

    # Calculate the ideal midpoint for each weather parameter
    temp_ideal = (temp_min + temp_max) / 2
    humidity_ideal = (humidity_min + humidity_max) / 2
    rainfall_ideal = (rainfall_min + rainfall_max) / 2

    # Calculate temperature suitability
    temp_range = (temp_max - temp_min) / 2
    temp_score = max(
        0,
        1 - abs(temperature - temp_ideal) / temp_range
    )

    # Calculate humidity suitability
    humidity_range = (humidity_max - humidity_min) / 2
    humidity_score = max(
        0,
        1 - abs(humidity - humidity_ideal) / humidity_range
    )

    # Current API rainfall represents recent rainfall,
    # so give a neutral score when no rainfall is reported
    if rainfall == 0:
        rainfall_score = 0.5
    else:
        rainfall_range = (rainfall_max - rainfall_min) / 2
        rainfall_score = max(
            0,
            1 - abs(rainfall - rainfall_ideal) / rainfall_range
        )

    # Combine the three weather components
    final_score = (
        temp_score * 40 +
        humidity_score * 30 +
        rainfall_score * 30
    )

    # Return the final score rounded to two decimal places
    return round(final_score, 2)


# Calculate weather suitability for the actual Top-3 crops
weather_scores = get_crop_weather_scores(
    top_3_crops,
    temperature,
    humidity,
    rainfall
)

# Display the improved crop-wise weather scores
print("Improved Top-3 Crop Weather Suitability:")

for crop, score in weather_scores.items():
    print(crop.title(), ":", score, "/ 100")

In [ ]:
# Calculate weather suitability for multiple recommended crops
def get_crop_weather_scores(crops, temperature, humidity, rainfall):
    
    # Store weather suitability scores for each crop
    scores = {}

    # Calculate the score for every crop
    for crop in crops:
        scores[crop] = crop_weather_score(
            crop,
            temperature,
            humidity,
            rainfall
        )

    return scores


# Example top-3 crops from the crop recommendation model
top_3_crops = ["rice", "maize", "wheat"]

# Calculate weather suitability for the top-3 crops
weather_scores = get_crop_weather_scores(
    top_3_crops,
    temperature,
    humidity,
    rainfall
)

# Display the results
print("Crop-wise Weather Suitability:")

for crop, score in weather_scores.items():
    print(crop.title(), ":", score, "/ 100")

In [ ]:
# Get the Top-3 crop recommendations from the trained crop model
crop_probabilities = rf_model.predict_proba(farmer_input)[0]

# Get the crop names learned by the Random Forest model
crop_names = rf_model.classes_

# Sort crops according to their prediction probability
top_indices = np.argsort(crop_probabilities)[::-1][:3]

# Store Top-3 crops and their probabilities
top_3_recommendations = []

for index in top_indices:
    top_3_recommendations.append({
        "crop": crop_names[index],
        "probability": crop_probabilities[index]
    })

# Display the actual Top-3 recommendations
print("Top-3 Crop Recommendations:")

for item in top_3_recommendations:
    print(
        item["crop"].title(),
        ":",
        round(item["probability"] * 100, 2),
        "%"
    )

In [ ]:
# Extract the crop names from the Top-3 recommendations
top_3_crops = [
    item["crop"] for item in top_3_recommendations
]

# Calculate weather suitability for each Top-3 crop
weather_scores = get_crop_weather_scores(
    top_3_crops,
    temperature,
    humidity,
    rainfall
)

# Display crop-wise weather suitability
print("Top-3 Crop Weather Suitability:")

for crop, score in weather_scores.items():
    
    # Display unavailable scores clearly
    if score is None:
        print(crop.title(), ": Weather data not available")
    else:
        print(crop.title(), ":", score, "/ 100")

In [ ]:
#first layer
# Combine crop model probability with weather suitability
# to calculate an overall pre-sowing suitability score

combined_scores = []

for item in top_3_recommendations:

    # Get the crop name
    crop = item["crop"]

    # Get crop recommendation probability
    crop_probability = item["probability"] * 100

    # Get weather suitability score
    weather_score = weather_scores.get(crop)

    # Calculate the combined suitability score
    # Crop model gets 60% weight and weather gets 40% weight
    final_score = (
        crop_probability * 0.60 +
        weather_score * 0.40
    )

    # Store the result
    combined_scores.append({
        "crop": crop,
        "crop_probability": round(crop_probability, 2),
        "weather_score": round(weather_score, 2),
        "combined_score": round(final_score, 2)
    })

# Sort crops according to the combined suitability score
combined_scores = sorted(
    combined_scores,
    key=lambda x: x["combined_score"],
    reverse=True
)

# Display the combined results
print("Crop + Weather Suitability:")

for item in combined_scores:
    print(
        item["crop"].title(),
        "| Crop Probability:", item["crop_probability"], "%",
        "| Weather:", item["weather_score"], "/ 100",
        "| Combined:", item["combined_score"], "/ 100"
    )

In [ ]:
# Create irrigation input for the currently recommended crop
# These values represent the current field and weather conditions

irrigation_input = pd.DataFrame([{
    "Soil_Type": "Loamy",
    "Crop_Type": "Rice",
    "Crop_Growth_Stage": "Sowing",
    "Season": "Kharif",
    "Irrigation_Type": "Drip",
    "Water_Source": "Groundwater",
    "Mulching_Used": "No",
    
    # Soil and environmental conditions
    "Soil_pH": farmer_input["ph"].iloc[0],
    "Soil_Moisture": 40,
    "Organic_Carbon": 1.0,
    "Electrical_Conductivity": 0.5,
    "Temperature_C": temperature,
    "Humidity": humidity,
    "Rainfall_mm": rainfall,
    "Sunlight_Hours": 7,
    "Wind_Speed_kmh": wind_speed * 3.6,
    "Field_Area_hectare": 1.0,
    "Previous_Irrigation_mm": 20,
    
    # Region information
    "Region": "South"
}])

# Display the irrigation input used for prediction
print("Irrigation Input Created Successfully!")
print(irrigation_input)

In [ ]:
# Predict the irrigation requirement for the current field conditions
irrigation_prediction = irrigation_model_v2.predict(irrigation_input)[0]

# Get prediction probabilities for each irrigation category
irrigation_probabilities = irrigation_model_v2.predict_proba(irrigation_input)[0]

# Get the irrigation classes from the trained model
irrigation_classes = irrigation_model_v2.classes_

# Display the predicted irrigation requirement
print("Predicted Irrigation Need:", irrigation_prediction)

# Display probability for each irrigation category
print("\nIrrigation Probabilities:")

for category, probability in zip(
    irrigation_classes,
    irrigation_probabilities
):
    print(category, ":", round(probability * 100, 2), "%")

In [ ]:
# Convert the predicted irrigation requirement into
# an irrigation resource-feasibility score
# Higher score means lower irrigation burden

irrigation_feasibility_map = {
    "Low": 100,
    "Medium": 75,
    "High": 50
}

# Get the irrigation feasibility score
irrigation_score = irrigation_feasibility_map[irrigation_prediction]

# Calculate the final Decision Engine score
# Crop suitability = 50%
# Weather suitability = 30%
# Irrigation feasibility = 20%

for item in combined_scores:

    # Get the crop and its existing combined score
    crop = item["crop"]
    
    # Get the crop + weather score
    crop_weather_score_value = item["combined_score"]

    # Combine all three decision factors
    final_decision_score = (
        crop_weather_score_value * 0.80 +
        irrigation_score * 0.20
    )

    # Store the final score
    item["irrigation_score"] = irrigation_score
    item["final_score"] = round(final_decision_score, 2)


# Rank crops according to the final Decision Engine score
combined_scores = sorted(
    combined_scores,
    key=lambda x: x["final_score"],
    reverse=True
)

# Display the final decision results
print("Final Decision Engine Results:\n")

for item in combined_scores:
    print(
        item["crop"].title(),
        "| Crop Probability:", item["crop_probability"], "%",
        "| Weather:", item["weather_score"], "/ 100",
        "| Irrigation:", item["irrigation_score"], "/ 100",
        "| Final Score:", item["final_score"], "/ 100"
    )

# Select the highest-scoring crop as the final recommendation
final_crop = combined_scores[0]["crop"]

print("\nFinal Recommended Crop:", final_crop.title())

In [ ]:
# Generate a human-readable explanation for the final crop recommendation
# using the outputs of the Crop Model, Weather Analysis, and Irrigation Model

final_result = combined_scores[0]

# Extract the values of the final recommended crop
recommended_crop = final_result["crop"].title()
crop_probability = final_result["crop_probability"]
weather_score = final_result["weather_score"]
irrigation_score = final_result["irrigation_score"]
final_score = final_result["final_score"]

# Create the explanation for the farmer
explanation = (
    f"{recommended_crop} is recommended because it received the "
    f"highest overall suitability score of {final_score}/100. "
    f"The crop recommendation model assigned it a probability of "
    f"{crop_probability}%, while the current weather suitability "
    f"score is {weather_score}/100. "
    f"The predicted irrigation requirement is {irrigation_prediction}, "
    f"with an irrigation feasibility score of {irrigation_score}/100."
)

# Display the final recommendation and explanation
print("Final Recommendation:", recommended_crop)
print("\nExplanation:")
print(explanation)

In [124]:
# Display the final Decision Engine output in a clean and structured format
# This combines crop recommendation, weather suitability, and irrigation analysis

print("=" * 60)
print("             AGROSENSE AI - DECISION ENGINE")
print("=" * 60)

# Display the final recommended crop
print(f"\nFinal Recommended Crop : {recommended_crop}")
print(f"Final Suitability Score: {final_score}/100")

# Display Top-3 crop comparison
print("\nTop-3 Crop Comparison")
print("-" * 60)

for rank, item in enumerate(combined_scores, start=1):
    print(
        f"{rank}. {item['crop'].title():15}"
        f" | Crop: {item['crop_probability']:6.2f}%"
        f" | Weather: {item['weather_score']:6.2f}"
        f" | Irrigation: {item['irrigation_score']:6.2f}"
        f" | Final: {item['final_score']:6.2f}"
    )

# Display current weather conditions
print("\nCurrent Weather Conditions")
print("-" * 60)
print(f"Temperature : {temperature} °C")
print(f"Humidity    : {humidity} %")
print(f"Rainfall    : {rainfall} mm")
print(f"Wind Speed  : {wind_speed} m/s")

# Display irrigation prediction
print("\nIrrigation Analysis")
print("-" * 60)
print(f"Predicted Irrigation Need: {irrigation_prediction}")

# Display the explanation generated by the system
print("\nWhy this crop was recommended")
print("-" * 60)
print(explanation)

print("\n" + "=" * 60)

             AGROSENSE AI - DECISION ENGINE

Final Recommended Crop : Rice
Final Suitability Score: 71.17/100

Top-3 Crop Comparison
------------------------------------------------------------
1. Rice            | Crop:  93.50% | Weather:  35.27 | Irrigation:  75.00 | Final:  71.17
2. Jute            | Crop:   6.50% | Weather:  36.82 | Irrigation:  75.00 | Final:  29.90
3. Pomegranate     | Crop:   0.00% | Weather:  23.27 | Irrigation:  75.00 | Final:  22.45

Current Weather Conditions
------------------------------------------------------------
Temperature : 34.2 °C
Humidity    : 68 %
Rainfall    : 0 mm
Wind Speed  : 5.14 m/s

Irrigation Analysis
------------------------------------------------------------
Predicted Irrigation Need: Medium

Why this crop was recommended
------------------------------------------------------------
Rice is recommended because it received the highest overall suitability score of 71.17/100. The crop recommendation model assigned it a probability of 93.5%